In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [3]:
data=pd.read_csv(r"loan_approval_data.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Applicant_ID        950 non-null    float64
 1   Applicant_Income    950 non-null    float64
 2   Coapplicant_Income  950 non-null    float64
 3   Employment_Status   950 non-null    object 
 4   Age                 950 non-null    float64
 5   Marital_Status      950 non-null    object 
 6   Dependents          950 non-null    float64
 7   Credit_Score        950 non-null    float64
 8   Existing_Loans      950 non-null    float64
 9   DTI_Ratio           950 non-null    float64
 10  Savings             950 non-null    float64
 11  Collateral_Value    950 non-null    float64
 12  Loan_Amount         950 non-null    float64
 13  Loan_Term           950 non-null    float64
 14  Loan_Purpose        950 non-null    object 
 15  Property_Area       950 non-null    object 
 16  Educati

In [ ]:
#differenciating columns
num_cols=data.select_dtypes(include="number").columns  # "float64","int64"
cat_cols=data.select_dtypes(include="object").columns

# Handling missing values

In [ ]:
# filling the null values
from sklearn.impute import SimpleImputer

no=SimpleImputer(strategy="mean")
data[num_cols]=no.fit_transform(data[num_cols])

cat=SimpleImputer(strategy="most_frequent")
data[cat_cols]=cat.fit_transform(data[cat_cols])

In [ ]:
data.drop(columns="Applicant_ID",inplace=True) # removed unrelated feature

In [ ]:
data.isnull().sum()
data.head()

# EDA

In [ ]:
#is our classes divided equally?

total=data["Loan_Approved"].value_counts()
plt.pie(total,autopct="%1.1f%%",labels=["NO","YES"])

In [ ]:
fig, axes = plt.subplots(2, 2,figsize=(9,6))


sns.histplot(ax=axes[0, 0], data=data, x="Employment_Status")
sns.histplot(ax=axes[0, 1], data=data, x="Gender")
sns.histplot(ax=axes[1, 0], data=data, x="Loan_Purpose")
sns.histplot(ax=axes[1, 1], data=data, x="Employer_Category")

plt.tight_layout()

In [ ]:
# checking outliers

fig,ax=plt.subplots(3,2,figsize=(9,9))

sns.boxplot(ax=ax[0,0],data=data,x="Loan_Approved",y="Applicant_Income")
sns.boxplot(ax=ax[0,1],data=data,x="Loan_Approved",y="Coapplicant_Income")
sns.boxplot(ax=ax[1,0],data=data,x="Loan_Approved",y="Credit_Score")
sns.boxplot(ax=ax[1,1],data=data,x="Loan_Approved",y="Existing_Loans")
sns.boxplot(ax=ax[2,0],data=data,x="Loan_Approved",y="DTI_Ratio")
sns.boxplot(ax=ax[2,1],data=data,x="Loan_Approved",y="Loan_Amount")

plt.tight_layout()
plt.show()

In [ ]:
#Loan_approved credit score and dti ratio

sns.histplot(data,x="Credit_Score",hue="Loan_Approved",bins=20,multiple="dodge")

In [ ]:
sns.histplot(data,x="DTI_Ratio",hue="Loan_Approved",bins=20,multiple="dodge")

In [ ]:
sns.histplot(data,x="Applicant_Income",hue="Loan_Approved",bins=20,multiple="dodge")

In [ ]:
data.info()
data.sample(5)

# feature encoding

In [ ]:
# Encoding categorical features

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data["Education_Level"]=le.fit_transform(data["Education_Level"])
data["Loan_Approved"]=le.fit_transform(data["Loan_Approved"])

In [ ]:
from sklearn.preprocessing import OneHotEncoder

OHE=OneHotEncoder(drop="first",sparse_output=False,handle_unknown='ignore')

cols=["Employment_Status","Marital_Status","Loan_Purpose","Property_Area","Gender","Employer_Category"]

encoded=OHE.fit_transform(data[cols])

In [ ]:
encoded_df=pd.DataFrame(encoded,columns=OHE.get_feature_names_out(cols),index=data.index)
data=pd.concat([data.drop(columns=cols),encoded_df],axis=1)

In [ ]:
data.info()

In [ ]:
#correlation heatmap

num_cols=data.select_dtypes(include="number")
corr_matrix=num_cols.corr()

plt.figure(figsize=(15,9))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

In [ ]:
num_cols.corr()["Loan_Approved"].sort_values(ascending=False)

In [ ]:
X=data.drop(columns=["Loan_Approved"])
y=data["Loan_Approved"]

# train_test_split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
X_test.head()

# Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
X_train_scaled

# Logistic Model

In [ ]:
#logistic model
from sklearn.linear_model import LogisticRegression

LRmodel=LogisticRegression()
LRmodel.fit(X_train_scaled,y_train)

y_pred=LRmodel.predict(X_test_scaled)

In [ ]:
#Evalution

from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score

print("Logistic Regression")
print("accuracy:",accuracy_score(y_test,y_pred))
print("precision:",precision_score(y_test,y_pred))
print("recall:",recall_score(y_test,y_pred))
print("confusion_matrix:",confusion_matrix(y_test,y_pred))

# KNN Model

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knnmodel=KNeighborsClassifier(n_neighbors=5)

knnmodel.fit(X_train_scaled,y_train)
y_predict=knnmodel.predict(X_test_scaled)

In [ ]:
print("KNN")
print("accuracy:",accuracy_score(y_test,y_predict))
print("precision:",precision_score(y_test,y_predict))
print("recall:",recall_score(y_test,y_predict))
print("confusion_matrix:",confusion_matrix(y_test,y_predict))

# Naive Bayes Model

In [ ]:
from sklearn.naive_bayes import GaussianNB

NBmodel=GaussianNB()

NBmodel.fit(X_train_scaled,y_train)
y_predict=NBmodel.predict(X_test_scaled)

print("Gaussian naive bayes")
print("accuracy:",accuracy_score(y_test,y_predict))
print("precision:",precision_score(y_test,y_predict))
print("recall:",recall_score(y_test,y_predict))
print("confusion_matrix:",confusion_matrix(y_test,y_predict))

# Best model by precision ==> Naive Bayes

# feature Engineering

In [ ]:
# feature Engineering 

data["credit_sq"]=data["Credit_Score"]**2
data["DTI_sq"]=data["DTI_Ratio"]**2

data["Applicant_Income_log"]=np.log1p(data["Applicant_Income"])

X=data.drop(columns=["Loan_Approved","Credit_Score","DTI_Ratio"])
y=data["Loan_Approved"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

sc=StandardScaler()

Xtr_scaled=sc.fit_transform(X_train)
Xte_scaled=sc.transform(X_test)

In [ ]:
X_train.head()

In [ ]:
#logistic model
from sklearn.linear_model import LogisticRegression

LRmodel=LogisticRegression()
LRmodel.fit(Xtr_scaled,y_train)

y_pred=LRmodel.predict(Xte_scaled)

print("Logistic Regression")
print("accuracy:",accuracy_score(y_test,y_pred))
print("precision:",precision_score(y_test,y_pred))
print("recall:",recall_score(y_test,y_pred))
print("confusion_matrix:",confusion_matrix(y_test,y_pred))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knnmodel=KNeighborsClassifier(n_neighbors=9)

knnmodel.fit(Xtr_scaled,y_train)
y_predict=knnmodel.predict(Xte_scaled)

print("KNN model")
print("accuracy:",accuracy_score(y_test,y_predict))
print("precision:",precision_score(y_test,y_predict))
print("recall:",recall_score(y_test,y_predict))
print("confusion_matrix:",confusion_matrix(y_test,y_predict))

# Best performed model

In [ ]:


from sklearn.naive_bayes import GaussianNB

NBmodel=GaussianNB()

NBmodel.fit(Xtr_scaled,y_train)
y_predict=NBmodel.predict(Xte_scaled)

print("Gaussian naive bayes")
print("accuracy:",accuracy_score(y_test,y_predict))
print("precision:",precision_score(y_test,y_predict))
print("recall:",recall_score(y_test,y_predict))
print("confusion_matrix:",confusion_matrix(y_test,y_predict))